In [ ]:
import subprocess
import json

import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

from api.FrameAndValidate import frame_cdif_document, validate_against_schema

In [4]:
BASE_DIR = "/home/dmlung/workspace/cdif-repos/cdif4xas/cdif-xas/"
RESOURCES_DIR = BASE_DIR + "resources"

MAPPER_JAR = BASE_DIR + "lib/" + "rmlmapper-8.1.0-r0-all.jar"
MAPPING_FILE = RESOURCES_DIR + "/mapping_dds.ttl"
OUTPUT_FILE = RESOURCES_DIR + "/cdif_dds.jsonld"
FRAMED_FILE = RESOURCES_DIR + "/cdif_dds_framed.jsonld"

FRAME_PATH = RESOURCES_DIR + "/CDIFDiscoveryDataDescriptionStructure-frame.jsonld"
CONTEXT_PATH = RESOURCES_DIR + "/context.json"
SCHEMA_PATH = RESOURCES_DIR + "/CDIFDiscoveryDataDescriptionStructureProfileStructuredSchema.json"

In [ ]:
# map
subprocess.run(
        ["java", "-jar", MAPPER_JAR, "-m", MAPPING_FILE, "-o", OUTPUT_FILE, "-s", "jsonld"],
        capture_output=True,
        text=True,
        check=True
    )

In [ ]:
# frame
framed = frame_cdif_document(OUTPUT_FILE, FRAME_PATH, CONTEXT_PATH)
with open(FRAMED_FILE, 'w', encoding='utf-8') as f:
    json.dump(framed, f, indent=2)

In [ ]:
# validate
with open(FRAMED_FILE, "r", encoding="utf-8") as f:
    framed = json.load(f)

    print("\nValidating against schema...")
    result = validate_against_schema(framed, SCHEMA_PATH)

    if result['valid']:
        print("Validation PASSED")
    else:
        print("Validation FAILED")
        print("\nErrors:")
        for error in result['errors']:
            path = '/'.join(str(p) for p in error.absolute_path) if error.absolute_path else '/'
            print(f"  - /{path}: {error.message}")
        sys.exit(1)